# A2 · Decisión ZAP (cielo)

**Spec:** [`docs/spec_A2_codex_sky_zap.md`](../docs/spec_A2_codex_sky_zap.md)  |  **Bloque:** A · Reducción  |  **Run por defecto:** `ROXs12b_realigned`

Decide y aplica (o descarta) la sustracción de cielo con ZAP.

| | |
|---|---|
| **Entrada** | Cubo reducido |
| **Salida (QC/productos)** | Cubo con cielo tratado (sin QC separado en este run) |
| **Consume aguas abajo** | A3, A4 |


## Qué es ZAP y por qué se necesita

**ZAP** (*Zurich Atmosphere Purge*, Soto et al. 2016) es una sustracción de **residuos de cielo** para MUSE basada en PCA. El pipeline (`muse_scipost subtract_sky`) ya resta un modelo de cielo, pero el **airglow** (líneas de OH y [O I] atmosféricas) es intenso y **varía en el tiempo** entre la exposición de ciencia y el modelo → suelen quedar **residuos de skylines**. ZAP construye una base PCA con los spaxels de **cielo** (con las fuentes enmascaradas) y elimina las componentes que describen esos residuos, dejando la señal astrofísica.

**Por qué importa aquí:** un residuo de skyline mal restado puede **imitar o contaminar** una línea espectral. Buscamos una línea débil de Hα en el compañero, así que el cielo residual es un contaminante de primer orden.

**El peligro (por qué NO se aplica a ciegas):** ZAP necesita suficientes spaxels de cielo *reales*. En el **campo diminuto de NFM**, con una estrella brillante y su compañero, la fracción de cielo es baja y las eigencomponentes pueden **absorber señal del compañero** — incluso *fabricar o borrar* una línea en Hα. Regla del proyecto: ante la duda, **no tocar la señal**.

**Decisión pre-registrada** (`musepipe.reduction.sky_zap.classify_zap_decision`), por métrica, no por juicio. `R` = RMS mediano en ventanas de skyline ÷ RMS mediano en continuo, medido en aperturas de cielo vacías:

| Condición | Decisión |
|---|---|
| `R ≤ 1.5` | **no necesario** → `zap_applied = False` |
| `R > 2.0` | **necesario** → `zap_applied = True` |
| `1.5 < R ≤ 2.0` | zona gris → checkpoint (no aplicar, preguntar) |
| fracción de cielo `< 0.25` | cielo insuficiente → checkpoint (no aplicar) |

Los parámetros de ZAP quedan en *default*; no se itera buscando 'el mejor resultado'.


## Cómo ejecutar de forma independiente

> ⚠️ **Etapa no re-ejecutable desde raw en este repo.** En la poda WP-10 se borraron los intermedios regenerables (`muse_scibasic`, `muse_scipost`, …). Se conservaron los productos finales y todo el QC. Este notebook **audita** el producto/QC existente y documenta el comando histórico.

Comando histórico (referencia, requiere los raw + `esorex`):

```bash
conda activate MUSE
bash scripts/sky_zap.sh
```


In [ ]:
import os, sys
# Añade notebooks/ (para _nbcommon) y la RAÍZ del repo (para importar musepipe),
# funcione el cwd en notebooks/ o en la raíz del repo.
_here = os.getcwd()
if os.path.basename(_here) != 'notebooks' and os.path.isdir(os.path.join(_here, 'notebooks')):
    _here = os.path.join(_here, 'notebooks')
for _p in (_here, os.path.dirname(_here)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
_root = str(nb.project_root())
if _root not in sys.path:
    sys.path.insert(0, _root)   # asegura 'import musepipe'
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))


## Auditar

Etapa de solo-auditoría: se carga el producto/QC más abajo.


## Resultados que llevaron a la conclusión

Métrica **M4 de cielo** del QC del cubo (`stages/stage00q_qc.json`) aplicada a la regla de decisión pre-registrada.


In [ ]:
qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
m4 = qc.get('m4_sky', {})
R = m4.get('R')
sky_frac = m4.get('sky_fraction')   # QCs antiguos no lo persisten (gap documentado)
print(f'm4_sky.R = {R}   (RMS skyline / RMS continuo en aperturas vacías)   [status {m4.get("status")}]')

# Decisión con la implementación OFICIAL (no reimplementada aquí):
try:
    from musepipe.reduction.sky_zap import classify_zap_decision
    if R is None:
        print('sin dato de R en el QC -> decisión no evaluable')
    else:
        if sky_frac is None:
            print('AVISO: el QC no persiste sky_fraction -> la rama '
                  'insufficient_sky (<0.25) no es re-derivable aquí; se evalúa solo la rama R.')
        d = classify_zap_decision(float(R), float(sky_frac) if sky_frac is not None else 1.0)
        print(f'classify_zap_decision: {d.decision}  ->  zap_applied = {d.zap_applied}'
              f'  (checkpoint_required={d.checkpoint_required})')
except Exception as e:
    print('No se pudo importar musepipe (kernel sin la pila científica):', type(e).__name__, e)
    print('Regla pre-registrada (espejo de classify_zap_decision): R<=1.5 no necesario | '
          'R>2.0 necesario | zona gris -> checkpoint | sky_fraction<0.25 -> checkpoint')

print()
print('Corroboración (nota del QC):')
print('  ', qc.get('note'))
print('M1/M2 se midieron del SKY_SPECTRUM cacheado (airglow, 32 exp,',
      qc.get('m1_wavelength', {}).get('n_measurements'), 'medidas) porque el')
print('cubo restado de cielo tiene <8 skylines usables.')


## De dónde sale R: los datos y la zona

**Un solo FITS:** `cube_telcorr.fits` (el producto de A1), extensión **DATA** (la STAT no interviene en R). R se mide sobre los **spaxels de cielo vacíos** de ese cubo — no hay varios archivos. *(Los 32 `SKY_SPECTRUM` cacheados son de M1/M2, no de R.)*

El gráfico reproduce M4 con las funciones canónicas (`compute_sky_residual_metrics`, `SKYLINE_WINDOWS`, `CONTINUUM_WINDOWS`):

- **Izquierda:** RMS por canal en las aperturas de cielo vs λ. En rojo las ventanas de skyline, en verde las de continuo; las punteadas son las medianas cuyo cociente es `R`. Se ven los residuos de OH en el rojo (>7200 Å), pero su mediana queda **por debajo** del continuo → R < 1.
- **Derecha:** la zona de cielo usada (azul) sobre la luz-blanca; el halo AO de la primaria queda excluido.

> Necesita el kernel **MUSE** (astropy) y el cubo en disco. Usa una máscara de cielo aproximada (percentil 30 de flujo), así que el R reproducido (~0.49) difiere levemente del oficial 0.547 (máscara de A4); la conclusión `R ≤ 1.5` es idéntica.


In [ ]:
MAKE_PLOT = True   # carga el cubo (~3.3 GB) vía astropy; requiere kernel MUSE
if MAKE_PLOT:
    try:
        import numpy as np
        import matplotlib.pyplot as plt
        from astropy.io import fits
        from musepipe.reduction.sky_zap import (
            compute_sky_residual_metrics, SKYLINE_WINDOWS, CONTINUUM_WINDOWS)

        qc = nb.load_qc('stages/stage00q_qc.json', RUN_ID)
        cube_path = qc.get('input_cube') or qc.get('cube', {}).get('file')
        print('FITS usado:', cube_path)

        h = fits.open(cube_path, memmap=True)
        data = np.asarray(h[1].data, dtype=np.float32)
        hd = h[1].header
        n3 = hd['NAXIS3']
        wave = hd['CRVAL3'] + (np.arange(n3) - (hd['CRPIX3'] - 1)) * hd['CD3_3']

        wl = np.nanmedian(data, axis=0)
        finite = np.isfinite(wl)
        thr = np.nanpercentile(wl[finite], 30)
        sky_mask = finite & (wl < thr)   # cielo vacio ~ spaxels mas debiles

        m = compute_sky_residual_metrics(data.astype(np.float64), wave, sky_mask)
        R = m['R_skyline_over_continuum']
        rms = m['channel_rms']
        print(f'R reproducido = {R:.3f}   (oficial m4_sky.R = {qc.get("m4_sky", {}).get("R")})')

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2),
                                       gridspec_kw={'width_ratios': [2, 1]})
        ax1.plot(wave, rms, lw=0.5, color='0.35')
        for i, (a, b) in enumerate(SKYLINE_WINDOWS):
            ax1.axvspan(a, b, color='tab:red', alpha=0.18,
                        label='ventana skyline' if i == 0 else None)
        for i, (a, b) in enumerate(CONTINUUM_WINDOWS):
            ax1.axvspan(a, b, color='tab:green', alpha=0.25,
                        label='ventana continuo' if i == 0 else None)
        ax1.axhline(m['skyline_rms_median'], color='tab:red', ls='--', lw=1)
        ax1.axhline(m['continuum_rms_median'], color='tab:green', ls='--', lw=1)
        ax1.set_xlabel('λ [Å]'); ax1.set_ylabel('RMS por canal (aperturas de cielo)')
        ax1.set_title(f'M4: RMS vs λ  →  R = med(skyline)/med(continuo) = {R:.3f}')
        ax1.set_ylim(0, np.nanpercentile(rms, 99)); ax1.legend(fontsize=8)

        ax2.imshow(np.log10(np.clip(wl, 1, None)), origin='lower', cmap='gray')
        ov = np.zeros((*wl.shape, 4)); ov[sky_mask] = [0.1, 0.5, 1.0, 0.5]
        ax2.imshow(ov, origin='lower')
        ax2.set_title('Zona de cielo (azul) sobre luz-blanca'); ax2.axis('off')
        fig.tight_layout()

        outdir = nb.run_dir(RUN_ID) / 'plots' / 'a2_m4'
        outdir.mkdir(parents=True, exist_ok=True)
        fig.savefig(outdir / 'm4_R.png', dpi=110)
        print('figura ->', outdir / 'm4_R.png')
        plt.show()
        h.close()
    except Exception as e:
        print('No se pudo generar el plot:', type(e).__name__, e)
        print('Necesita el kernel MUSE (astropy) y el cubo en disco (campo input_cube del QC).')


## Decisiones y notas
- La métrica M4 (`R`) y la caracterización del airglow viven en A4/`stage00q_qc.json`; la regla de decisión, en `musepipe/reduction/sky_zap.py`.


## Conclusión (registrada)

**Decisión: ZAP NO aplicado — `zap_applied = False` (`not_needed`).**

- **Fecha del análisis:** 2026-07-09 (QC A4/M4 sobre el cubo realineado, commit `700f009`); el cubo se redujo el 2026-07-08.
- **Datos:** cubo NFM-AO auto-reducido `cube_telcorr.fits` (OB 3444577, Prog 109.23B7.002, **7 exposiciones** MUSE.2022-09-01T00:36–02:03) + `SKY_SPECTRUM` cacheado (32 exposiciones) para caracterizar el airglow.
- **Evidencia:** `R = 0.547 ≤ 1.5` (umbral *no necesario*); el cubo restado de cielo tiene **<8 skylines usables** (residual al nivel de ruido). En el campo diminuto NFM, ZAP aportaría ~0 y arriesgaría absorber señal del compañero.
- **Estado M4 = yellow:** el residuo es bajo, pero la escasez de skylines hace la métrica menos robusta que en WFM. No bloqueante.
- **Para el paper:** registrar como decisión con su métrica (R=0.547), **no** como omisión. Nada es paper-válido hasta cerrar el A-block.
